# ANN topological simplification pipeline

This notebook trains the requested feedforward models, captures hidden activations on a fixed probe set, runs ripser, computes COM, and plots COM vs architecture size.

Pipeline order:
1. Training
2. Activation capture
3. Ripser
4. COM
5. Plotting


In [ ]:
# Cell 1: setup

# !pip3 -q install ripser dill scipy seaborn pandas scikit-learn

import os
import json
import math
import random
from pathlib import Path

import dill
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from ripser import ripser
from scipy.stats import pearsonr, spearmanr, kendalltau

# Reproducibility
GLOBAL_SEED = 0
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# Paths
ROOT = Path('${TDL_ROOT_DIR}/John/ANN')
DATA_PATH = ROOT / 'full_dataset.npz'
MODEL_ROOT = ROOT / 'models'
RIPSER_ROOT = ROOT / 'ripser_results'
FIG_ROOT = ROOT / 'figures'
LOG_ROOT = ROOT / 'logs'
MODEL_SIZES_JSON = ROOT / 'model_sizes.json'

for p in [MODEL_ROOT, RIPSER_ROOT, FIG_ROOT, LOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Experiment settings
N_SEEDS = 30
TARGET_ACC = 0.99
BATCH_SIZE = 256
MAX_EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 0.0
PATIENCE = 20

# Ripser / COM settings
ETA = 2.5
DIMS = (0,)
N_PERM = 15   # closest direct ripser analogue to the user's k = 15 request
USE_RUNNING_MIN = True
INCLUDE_OUTPUT = True
DIR_NAME = f'ripser_k{N_PERM}_eta{ETA}'


In [ ]:
# Cell 2: load data and build a fixed probe set

def load_npz_dataset(path: Path):
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)

    # Common conventions
    if {'X_train', 'y_train', 'X_test', 'y_test'} <= keys:
        X_train = data['X_train']
        y_train = data['y_train']
        X_test = data['X_test']
        y_test = data['y_test']
        return (X_train, y_train), (X_test, y_test)

    if {'X', 'y'} <= keys:
        X = data['X']
        y = data['y']
        return (X, y), None

    # Fallback: try to infer the two largest arrays as X and y
    arrays = [(k, data[k]) for k in data.files if isinstance(data[k], np.ndarray)]
    if len(arrays) < 2:
        raise ValueError(f'Could not infer dataset arrays from {path}; found keys: {data.files}')

    arrays = sorted(arrays, key=lambda kv: kv[1].size, reverse=True)
    X = arrays[0][1]
    y = arrays[1][1]
    return (X, y), None

(dataset_train, dataset_test) = load_npz_dataset(DATA_PATH)
X_all, y_all = dataset_train

X_all = np.asarray(X_all)
y_all = np.asarray(y_all)

# Convert labels to integer class indices if needed
if y_all.ndim > 1:
    y_all = y_all.argmax(axis=-1)
y_all = y_all.astype(np.int64)

# Basic shape normalization
if X_all.ndim == 2:
    # [N, D] already fine
    pass
elif X_all.ndim == 3:
    # [N, C, L] or similar; flatten to vectors for a vanilla MLP
    X_all = X_all.reshape(X_all.shape[0], -1)
else:
    X_all = X_all.reshape(X_all.shape[0], -1)

print('X shape:', X_all.shape)
print('y shape:', y_all.shape)
print('num classes:', int(np.max(y_all)) + 1)

# Fixed probe set used for activation capture and ripser
# Keep this identical across every model.
PROBE_SIZE = min(1024, len(X_all))
probe_idx = np.random.RandomState(GLOBAL_SEED).choice(len(X_all), size=PROBE_SIZE, replace=False)
X_probe = X_all[probe_idx]
y_probe = y_all[probe_idx]

# Train/val split
N = len(X_all)
N_VAL = int(0.1 * N)
N_TRAIN = N - N_VAL
train_ds_full = TensorDataset(torch.tensor(X_all, dtype=torch.float32), torch.tensor(y_all, dtype=torch.long))
train_ds, val_ds = random_split(
    train_ds_full,
    [N_TRAIN, N_VAL],
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
probe_loader = DataLoader(TensorDataset(torch.tensor(X_probe, dtype=torch.float32)), batch_size=BATCH_SIZE, shuffle=False)

input_dim = X_all.shape[1]
num_classes = int(np.max(y_all)) + 1
print('input_dim:', input_dim)
print('num_classes:', num_classes)
print('probe size:', PROBE_SIZE)


In [ ]:
# Cell 3: model grid

ARCHS = {
    '30x8': [30, 30, 30, 30, 30, 30, 30, 30],
    '24x8': [24, 24, 24, 24, 24, 24, 24, 24],
    '18x8': [18, 18, 18, 18, 18, 18, 18, 18],
    '30x4_24x4': [30, 30, 30, 30, 24, 24, 24, 24],
    '30x4_18x4': [30, 30, 30, 30, 18, 18, 18, 18],
    '30x4_12x4': [30, 30, 30, 30, 12, 12, 12, 12],
}

ACTIVATIONS = ['relu', 'tanh', 'leaky_relu']

MODEL_SIZES = {name: int(sum(dims)) for name, dims in ARCHS.items()}
with open(MODEL_SIZES_JSON, 'w') as f:
    json.dump(MODEL_SIZES, f, indent=2)

print(MODEL_SIZES)

# Save a small manifest for convenience
manifest = pd.DataFrame([
    {'arch': k, 'hidden_dims': v, 'hidden_sum': sum(v)} for k, v in ARCHS.items()
])
manifest


In [ ]:
# Cell 4: generic feedforward model and training utilities

class FeedForwardNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, activation_name='relu'):
        super().__init__()
        act_map = {
            'relu': nn.ReLU,
            'tanh': nn.Tanh,
            'leaky_relu': nn.LeakyReLU,
        }
        if activation_name not in act_map:
            raise ValueError(f'Unknown activation: {activation_name}')
        act_cls = act_map[activation_name]

        layers = []
        dims = [input_dim] + list(hidden_dims)
        for i in range(len(hidden_dims)):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(act_cls())
        layers.append(nn.Linear(dims[-1], num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_model(hidden_dims, activation_name):
    return FeedForwardNet(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        activation_name=activation_name,
    )


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=-1)
    return (preds == y).float().mean().item()


def evaluate(model, loader, device=DEVICE):
    model.eval()
    total_correct = 0
    total = 0
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(xb)
            total_correct += (logits.argmax(dim=-1) == yb).sum().item()
            total += len(xb)
    return {'loss': total_loss / max(total, 1), 'acc': total_correct / max(total, 1)}


def train_one_model(hidden_dims, activation_name, seed, save_path, max_epochs=MAX_EPOCHS, target_acc=TARGET_ACC):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = build_model(hidden_dims, activation_name).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = -1.0
    best_state = None
    best_epoch = -1
    patience_left = PATIENCE
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(xb)
            train_correct += (logits.argmax(dim=-1) == yb).sum().item()
            train_total += len(xb)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_metrics = evaluate(model, val_loader)

        row = {
            'seed': seed,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
        }
        history.append(row)

        if val_metrics['acc'] > best_val_acc:
            best_val_acc = val_metrics['acc']
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1

        if best_val_acc >= target_acc:
            break
        if patience_left <= 0:
            break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.to('cpu')
    model.eval()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'state_dict': model.state_dict(),
        'hidden_dims': hidden_dims,
        'activation_name': activation_name,
        'seed': seed,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
    }, save_path)

    return model, pd.DataFrame(history)


In [ ]:
# Cell 5: train the full grid and save checkpoints

# Expected output layout:
# models/<arch>/<activation>/seed_<seed>.pth

train_summaries = []

for arch_name, hidden_dims in ARCHS.items():
    for act_name in ACTIVATIONS:
        for seed in range(N_SEEDS):
            ckpt_path = MODEL_ROOT / arch_name / act_name / f'seed_{seed}.pth'
            print(f'Training {arch_name} | {act_name} | seed {seed}')
            model, hist = train_one_model(hidden_dims, act_name, seed, ckpt_path)

            final_val = evaluate(model.to(DEVICE), val_loader)
            final_train = evaluate(model.to(DEVICE), train_loader)
            model.to('cpu')

            hist_path = LOG_ROOT / arch_name / act_name / f'seed_{seed}.csv'
            hist_path.parent.mkdir(parents=True, exist_ok=True)
            hist.to_csv(hist_path, index=False)

            train_summaries.append({
                'arch': arch_name,
                'activation': act_name,
                'seed': seed,
                'checkpoint': str(ckpt_path),
                'train_acc': final_train['acc'],
                'val_acc': final_val['acc'],
                'train_loss': final_train['loss'],
                'val_loss': final_val['loss'],
            })

summary_df = pd.DataFrame(train_summaries)
summary_df.head()


In [ ]:
# Cell 6: activation capture helpers

def load_trained_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    hidden_dims = ckpt['hidden_dims']
    activation_name = ckpt['activation_name']
    model = build_model(hidden_dims, activation_name)
    model.load_state_dict(ckpt['state_dict'], strict=True)
    model.eval()
    return model, hidden_dims, activation_name


def get_hidden_linear_names(model):
    # All Linear layers except the final classifier layer
    names = []
    modules = list(model.named_modules())
    linear_names = [name for name, module in modules if isinstance(module, nn.Linear)]
    if len(linear_names) < 2:
        raise ValueError('Model must have at least one hidden Linear layer and one output Linear layer.')
    return linear_names[:-1]


@torch.no_grad()
def capture_hidden_activations(model, loader, layer_names):
    modules = dict(model.named_modules())
    for name in layer_names:
        if name not in modules:
            raise KeyError(f'Layer {name} not found in model. Available names include: {list(modules.keys())[:20]}')

    buffers = {name: [] for name in layer_names}
    handles = []

    def make_hook(layer_name):
        def hook(module, inp, out):
            y = out.detach().cpu()
            if y.ndim > 2:
                y = y.reshape(y.shape[0], -1)
            buffers[layer_name].append(y.numpy())
        return hook

    for name in layer_names:
        handles.append(modules[name].register_forward_hook(make_hook(name)))

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            _ = model.to(DEVICE)(xb)

    for h in handles:
        h.remove()

    activations = {name: np.concatenate(buffers[name], axis=0) for name in layer_names}
    return activations


In [ ]:
# Cell 7: capture activations for every checkpoint

ACT_ROOT = ROOT / 'activations'
ACT_ROOT.mkdir(parents=True, exist_ok=True)

activation_manifest = []

for arch_name in ARCHS:
    for act_name in ACTIVATIONS:
        ckpt_dir = MODEL_ROOT / arch_name / act_name
        if not ckpt_dir.exists():
            print('Skipping missing', ckpt_dir)
            continue

        for seed in range(N_SEEDS):
            ckpt_path = ckpt_dir / f'seed_{seed}.pth'
            if not ckpt_path.exists():
                print('Missing checkpoint:', ckpt_path)
                continue

            model, hidden_dims, activation_name = load_trained_model(ckpt_path)
            layer_names = get_hidden_linear_names(model)
            acts = capture_hidden_activations(model, probe_loader, layer_names)

            out_dir = ACT_ROOT / arch_name / act_name / f'seed_{seed}'
            out_dir.mkdir(parents=True, exist_ok=True)

            # Save the fixed probe input as the baseline 'input layer' point cloud
            with open(out_dir / 'input_layer.pkl', 'wb') as f:
                dill.dump(X_probe, f)

            for lname, A in acts.items():
                safe_name = lname.replace('.', '_')
                with open(out_dir / f'{safe_name}.pkl', 'wb') as f:
                    dill.dump(A, f)

            activation_manifest.append({
                'arch': arch_name,
                'activation': act_name,
                'seed': seed,
                'checkpoint': str(ckpt_path),
                'activation_dir': str(out_dir),
                'layer_names': layer_names,
            })

activation_manifest_df = pd.DataFrame(activation_manifest)
activation_manifest_df.head()


In [ ]:
# Cell 8: ripser utilities

def standardize_features(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    mu = x.mean(axis=0, keepdims=True)
    sd = x.std(axis=0, keepdims=True)
    return (x - mu) / (sd + eps)


def compute_diagrams_for_point_cloud(X, maxdim=1, standardize=True, n_perm=15):
    X = np.asarray(X, dtype=np.float32)
    ok = np.isfinite(X).all(axis=1)
    X = X[ok]
    if X.shape[0] < 3:
        raise ValueError('Need at least 3 points for ripser.')
    if standardize:
        X = standardize_features(X)
    return ripser(X, maxdim=maxdim, n_perm=n_perm)['dgms']


def save_diagrams(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        dill.dump(obj, f)


def load_pickle(path):
    with open(path, 'rb') as f:
        return dill.load(f)


In [ ]:
# Cell 9: run ripser on every captured activation set

RIP_ROOT = RIPSER_ROOT / DIR_NAME
RIP_ROOT.mkdir(parents=True, exist_ok=True)

ripser_manifest = []

for row in activation_manifest:
    arch_name = row['arch']
    act_name = row['activation']
    seed = row['seed']
    act_dir = Path(row['activation_dir'])

    out_dir = RIP_ROOT / arch_name / act_name / f'seed_{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)

    input_cloud = load_pickle(act_dir / 'input_layer.pkl')
    input_dgm = compute_diagrams_for_point_cloud(
        input_cloud,
        maxdim=1,
        standardize=True,
        n_perm=N_PERM,
    )
    save_diagrams(input_dgm, out_dir / 'input_layer.pkl')

    layer_files = sorted([p for p in act_dir.glob('*.pkl') if p.name != 'input_layer.pkl'])
    layer_diagrams = []

    for p in layer_files:
        A = load_pickle(p)
        dgm = compute_diagrams_for_point_cloud(
            A,
            maxdim=1,
            standardize=True,
            n_perm=N_PERM,
        )
        layer_diagrams.append(dgm)

    save_diagrams(layer_diagrams, out_dir / 'model.pkl')

    ripser_manifest.append({
        'arch': arch_name,
        'activation': act_name,
        'seed': seed,
        'ripser_dir': str(out_dir),
        'num_layers': len(layer_diagrams),
    })

ripser_manifest_df = pd.DataFrame(ripser_manifest)
ripser_manifest_df.head()
